# Inference and Export

Run this notebook after training in `01_potsdam_windowed_segmentation.ipynb`.

This notebook uses the framework's built-in inference pipeline:
- `InferenceCSVBuilder` — builds a CSV of test images
- `MultiClassInferenceProcessor` — sliding-window inference with tile merging
- `RasterExportInferenceStrategy` — saves predictions as GeoTIFF

**Steps:**
1. Load a trained checkpoint
2. Build the test CSV
3. Run inference on a single patch (sanity check)
4. Run full-scene sliding-window inference on one test tile
5. Visualize predictions vs ground truth
6. Compute per-class IoU

---
## Section 1 — Setup

### 1.1 — Standard imports

In [ ]:
from pathlib import Path

### 1.2 — Scientific imports

In [ ]:
import numpy as np
import torch
import rasterio
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap

### 1.3 — Framework imports

In [ ]:
from pytorch_segmentation_models_trainer.model_loader.model import Model
from pytorch_segmentation_models_trainer.tools.inference.inference_csv_builder import (
    InferenceCSVBuilder,
)
from pytorch_segmentation_models_trainer.tools.inference.inference_processors import (
    MultiClassInferenceProcessor,
)
from pytorch_segmentation_models_trainer.tools.inference.export_inference import (
    RasterExportInferenceStrategy,
)

### 1.4 — Potsdam class definitions

In [ ]:
CLASS_NAMES = [
    'Impervious surfaces',
    'Building',
    'Low vegetation',
    'Tree',
    'Car',
    'Clutter / background',
]
CLASS_COLORS_RGB = [
    (255, 255, 255),
    (0,   0,   255),
    (0,   255, 255),
    (0,   255, 0  ),
    (255, 255, 0  ),
    (255, 0,   0  ),
]
CLASS_COLORS_NORM = [(r/255, g/255, b/255) for r, g, b in CLASS_COLORS_RGB]
CMAP = ListedColormap(CLASS_COLORS_NORM)
NUM_CLASSES = len(CLASS_NAMES)
print(f'{NUM_CLASSES} classes configured.')

### 1.5 — Configure paths

**Edit these before running.**

In [ ]:
# ─── Edit these ──────────────────────────────────────────────────────────────
CHECKPOINT_PATH = Path('/data/checkpoints/best.ckpt')
POTSDAM_ROOT    = Path('/data/potsdam')
OUTPUT_DIR      = POTSDAM_ROOT / 'predictions'
# ─────────────────────────────────────────────────────────────────────────────

TEST_IMAGES_DIR = POTSDAM_ROOT / 'test' / 'images'
TEST_MASKS_DIR  = POTSDAM_ROOT / 'test' / 'masks'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('Checkpoint :', CHECKPOINT_PATH)
print('Test images:', TEST_IMAGES_DIR)
print('Output dir :', OUTPUT_DIR)

### 1.6 — Verify checkpoint exists

In [ ]:
assert CHECKPOINT_PATH.exists(), f'Checkpoint not found: {CHECKPOINT_PATH}'
print(f'Checkpoint found ({CHECKPOINT_PATH.stat().st_size / 1e6:.1f} MB).')

### 1.7 — Select device

In [ ]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', DEVICE)

---
## Section 2 — Load the Model

### 2.1 — Load checkpoint

In [ ]:
model = Model.load_from_checkpoint(str(CHECKPOINT_PATH), map_location=DEVICE)
print('Model loaded:', type(model).__name__)

### 2.2 — Switch to eval mode

In [ ]:
model.eval()
model.to(DEVICE)
print('Model in eval mode on', DEVICE)

### 2.3 — Count model parameters

In [ ]:
total_params = sum(p.numel() for p in model.parameters())
print(f'Total parameters: {total_params:,}')

---
## Section 3 — Build the Test CSV

`InferenceCSVBuilder` scans the test images folder and records each tile's path and dimensions.  
This CSV is the input to `TiledInferenceImageDataset` and `MultiClassInferenceProcessor`.

### 3.1 — Build the test manifest

In [ ]:
test_csv_path = POTSDAM_ROOT / 'test_inference.csv'

builder = InferenceCSVBuilder(
    images_folder=str(TEST_IMAGES_DIR),
    image_pattern='*.tif',
    recursive=False,
    masks_folder=str(TEST_MASKS_DIR),
    mask_pattern='*.tif',
    mask_suffix='',
    root_dir=str(POTSDAM_ROOT),
)

df_test = builder.build_csv(str(test_csv_path))
df_test.to_csv(test_csv_path, index=False)

print(f'{len(df_test)} test tiles  →  {test_csv_path}')

### 3.2 — Preview the test manifest

In [ ]:
df_test.head()

### 3.3 — Select the first test tile

In [ ]:
test_row       = df_test.iloc[0]
test_image_path = Path(test_row['image'])
test_mask_path  = Path(test_row['mask'])

print('Test image:', test_image_path.name)
print('GT mask   :', test_mask_path.name)

---
## Section 4 — Single-Patch Sanity Check

Before running full-scene inference, verify that a single 256×256 patch flows through the model correctly.

### 4.1 — Crop one patch from the test image

In [ ]:
PATCH_SIZE = 256

with rasterio.open(test_image_path) as src:
    patch_np = src.read()[:3, :PATCH_SIZE, :PATCH_SIZE]  # (3, H, W)

print(f'Patch shape : {patch_np.shape}')
print(f'Value range : [{patch_np.min()}, {patch_np.max()}]')

### 4.2 — Normalize and run forward pass

In [ ]:
patch_float  = patch_np.astype(np.float32) / 255.0
patch_tensor = torch.from_numpy(patch_float).unsqueeze(0).to(DEVICE)  # (1, 3, H, W)

with torch.no_grad():
    logits = model(patch_tensor)  # (1, NUM_CLASSES, H, W)

print(f'Logits shape: {logits.shape}')

### 4.3 — Convert logits to class map

In [ ]:
pred_patch = logits.argmax(dim=1).squeeze(0).cpu().numpy()  # (H, W)
print(f'Prediction shape : {pred_patch.shape}')
print(f'Classes predicted: {np.unique(pred_patch)}')

### 4.4 — Visualize the single-patch prediction

In [ ]:
img_display = np.clip(patch_np.transpose(1, 2, 0), 0, 255).astype(np.uint8)

legend_patches = [
    mpatches.Patch(color=CLASS_COLORS_NORM[i], label=f'{i} — {CLASS_NAMES[i]}')
    for i in range(NUM_CLASSES)
]

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(img_display)
axes[0].set_title('Input patch (256×256)')
axes[0].axis('off')

axes[1].imshow(pred_patch, cmap=CMAP, vmin=0, vmax=NUM_CLASSES - 1)
axes[1].set_title('Predicted mask')
axes[1].axis('off')
axes[1].legend(handles=legend_patches, loc='upper right', fontsize=8, framealpha=0.8)

plt.tight_layout()
plt.show()

---
## Section 5 — Full-Scene Inference with `MultiClassInferenceProcessor`

`MultiClassInferenceProcessor.make_inference()` handles:
- Albumentations normalization
- Padding to a multiple of the tile size
- Sliding-window tiling via `pytorch-toolbelt` `ImageSlicer`
- Batched forward passes
- `TileMerger` score fusion (averages overlapping logits)
- Argmax to class map

### 5.1 — Configure inference parameters

In [ ]:
MODEL_INPUT_SHAPE = [256, 256]   # must match training patch size
STEP_SHAPE        = [128, 128]   # 50% overlap for smoother borders
BATCH_SIZE        = 8            # tiles per GPU batch

print(f'Tile size  : {MODEL_INPUT_SHAPE}')
print(f'Step       : {STEP_SHAPE}  ({100*(1 - STEP_SHAPE[0]/MODEL_INPUT_SHAPE[0]):.0f}% overlap)')
print(f'Batch size : {BATCH_SIZE}')

### 5.2 — Set up the export strategy

`RasterExportInferenceStrategy` saves the prediction as a GeoTIFF with the source image's CRS and transform preserved.

In [ ]:
output_pred_path = OUTPUT_DIR / f'{test_image_path.stem}_prediction.tif'

export_strategy = RasterExportInferenceStrategy(
    output_file_path=str(output_pred_path),
)

print('Predictions will be saved to:', output_pred_path)

### 5.3 — Create the inference processor

In [ ]:
processor = MultiClassInferenceProcessor(
    model=model,
    device=DEVICE,
    batch_size=BATCH_SIZE,
    export_strategy=export_strategy,
    model_input_shape=MODEL_INPUT_SHAPE,
    step_shape=STEP_SHAPE,
    num_classes=NUM_CLASSES,
)

print('Processor created:', type(processor).__name__)

### 5.4 — Run full-scene inference on the test tile

`processor.process()` reads the image, tiles it, runs inference, merges scores, saves the GeoTIFF.

In [ ]:
print(f'Running inference on: {test_image_path.name} …')
processor.process(str(test_image_path))
print(f'Done. Saved to: {output_pred_path}')

### 5.5 — Verify the output file

In [ ]:
assert output_pred_path.exists(), 'Output file not found!'

with rasterio.open(output_pred_path) as src:
    pred_full = src.read(1)  # (H, W)
    print(f'Prediction shape : {pred_full.shape}')
    print(f'CRS              : {src.crs}')
    print(f'Classes in output: {np.unique(pred_full)}')

---
## Section 6 — Visualize Predictions vs Ground Truth

### 6.1 — Load ground-truth mask and input image

In [ ]:
with rasterio.open(test_image_path) as src:
    full_img = np.clip(src.read()[:3].transpose(1, 2, 0), 0, 255).astype(np.uint8)

with rasterio.open(test_mask_path) as src:
    gt_mask = src.read(1)

print(f'Image shape : {full_img.shape}')
print(f'GT mask shape: {gt_mask.shape}')
print(f'Pred shape  : {pred_full.shape}')

### 6.2 — Side-by-side: image / ground truth / prediction

In [ ]:
legend_patches = [
    mpatches.Patch(color=CLASS_COLORS_NORM[i], label=f'{i} — {CLASS_NAMES[i]}')
    for i in range(NUM_CLASSES)
]

fig, axes = plt.subplots(1, 3, figsize=(21, 7))

axes[0].imshow(full_img)
axes[0].set_title('Input image')
axes[0].axis('off')

axes[1].imshow(gt_mask, cmap=CMAP, vmin=0, vmax=NUM_CLASSES - 1)
axes[1].set_title('Ground truth')
axes[1].axis('off')

axes[2].imshow(pred_full, cmap=CMAP, vmin=0, vmax=NUM_CLASSES - 1)
axes[2].set_title('Prediction (MultiClassInferenceProcessor)')
axes[2].axis('off')

fig.legend(handles=legend_patches, loc='lower center', ncol=3, fontsize=9, framealpha=0.8)
plt.tight_layout(rect=[0, 0.07, 1, 1])
plt.show()

### 6.3 — Difference map: where prediction differs from ground truth

In [ ]:
# Crop to the smaller shape if sizes differ slightly due to padding
h = min(gt_mask.shape[0], pred_full.shape[0])
w = min(gt_mask.shape[1], pred_full.shape[1])
diff_map = (gt_mask[:h, :w] != pred_full[:h, :w]).astype(np.uint8)

error_pct = diff_map.mean() * 100
print(f'Pixel error rate: {error_pct:.2f}%')

fig, ax = plt.subplots(figsize=(8, 8))
ax.imshow(diff_map, cmap='Reds', vmin=0, vmax=1)
ax.set_title(f'Error map (red = wrong)  —  {error_pct:.1f}% error')
ax.axis('off')
plt.tight_layout()
plt.show()

---
## Section 7 — Per-Class IoU

### 7.1 — Compute IoU for each class

In [ ]:
gt_crop   = gt_mask[:h, :w]
pred_crop = pred_full[:h, :w]

iou_scores = []
for cls_idx in range(NUM_CLASSES):
    pred_c = pred_crop == cls_idx
    gt_c   = gt_crop   == cls_idx
    intersection = int((pred_c & gt_c).sum())
    union        = int((pred_c | gt_c).sum())
    iou = intersection / union if union > 0 else float('nan')
    iou_scores.append(iou)

### 7.2 — Print IoU table

In [ ]:
print(f'{'Class':<25s}  IoU')
print('─' * 36)
for i, (name, iou) in enumerate(zip(CLASS_NAMES, iou_scores)):
    iou_str = f'{iou:.4f}' if not np.isnan(iou) else '  n/a '
    print(f'[{i}] {name:<22s}  {iou_str}')

valid = [s for s in iou_scores if not np.isnan(s)]
print('─' * 36)
print(f'mIoU                    :  {np.mean(valid):.4f}')

### 7.3 — Plot IoU bar chart

In [ ]:
bar_heights = [v if not np.isnan(v) else 0.0 for v in iou_scores]

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar(CLASS_NAMES, bar_heights, color=CLASS_COLORS_NORM, edgecolor='grey', linewidth=0.5)

miou = np.mean(valid)
ax.axhline(miou, color='black', linestyle='--', linewidth=1, label=f'mIoU = {miou:.4f}')
ax.legend()
ax.set_ylim(0, 1.05)
ax.set_ylabel('IoU')
ax.set_title(f'Per-class IoU — {test_image_path.name}')
ax.tick_params(axis='x', rotation=20)
plt.tight_layout()
plt.show()

---
## Section 8 — Run Inference on All Test Tiles

### 8.1 — Loop over all test tiles

In [ ]:
all_iou_scores = []

for _, row in df_test.iterrows():
    img_path  = Path(row['image'])
    msk_path  = Path(row['mask'])
    out_path  = OUTPUT_DIR / f'{img_path.stem}_prediction.tif'

    # Create a fresh export strategy pointing to this tile's output
    tile_export = RasterExportInferenceStrategy(output_file_path=str(out_path))
    tile_proc   = MultiClassInferenceProcessor(
        model=model,
        device=DEVICE,
        batch_size=BATCH_SIZE,
        export_strategy=tile_export,
        model_input_shape=MODEL_INPUT_SHAPE,
        step_shape=STEP_SHAPE,
        num_classes=NUM_CLASSES,
    )

    print(f'Processing: {img_path.name} …', end=' ', flush=True)
    tile_proc.process(str(img_path))

    # Compute IoU against ground truth
    with rasterio.open(out_path) as src:
        pred = src.read(1)
    with rasterio.open(msk_path) as src:
        gt = src.read(1)

    h = min(gt.shape[0], pred.shape[0])
    w = min(gt.shape[1], pred.shape[1])
    ious = []
    for cls in range(NUM_CLASSES):
        inter = int(((pred[:h, :w] == cls) & (gt[:h, :w] == cls)).sum())
        union = int(((pred[:h, :w] == cls) | (gt[:h, :w] == cls)).sum())
        ious.append(inter / union if union > 0 else float('nan'))
    all_iou_scores.append(ious)

    valid_tile = [s for s in ious if not np.isnan(s)]
    print(f'mIoU = {np.mean(valid_tile):.4f}')

### 8.2 — Aggregate mean IoU across all test tiles

In [ ]:
all_iou_arr = np.array(all_iou_scores)  # (N_tiles, N_classes)
mean_iou_per_class = np.nanmean(all_iou_arr, axis=0)

print('Mean IoU per class across all test tiles:')
print(f'{'Class':<25s}  Mean IoU')
print('─' * 38)
for i, (name, iou) in enumerate(zip(CLASS_NAMES, mean_iou_per_class)):
    print(f'[{i}] {name:<22s}  {iou:.4f}')
print('─' * 38)
print(f'mIoU (all classes)        :  {np.nanmean(mean_iou_per_class):.4f}')

---
## Summary

| Step | Framework class used |
|------|---------------------|
| Build test CSV | `InferenceCSVBuilder` |
| Single-patch sanity check | `model.forward()` directly |
| Full-scene inference | `MultiClassInferenceProcessor.process()` |
| GeoTIFF export | `RasterExportInferenceStrategy` |
| Per-class IoU | Manual, per-tile and aggregated |

The exported GeoTIFFs in `predictions/` can be opened in QGIS or ArcGIS.